# 04. 버전별 F1 모아 보기

| 항목 | 내용 |
| --- | --- |
| 입력 | `03_model_f1.ipynb`를 실행해 만든 `data/processed/model_results/*_summary.csv` |
| 출력 | 화면의 표 (저장하지 않음) |
| 보는 법 | 행은 데이터 버전, 열은 모델, 값은 F1 |

## 1. 공통 경로

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v6)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV (이름만 바꾸면 됨)
DATA_FILE = "cal01_base_v1.csv"  # 04는 모델 결과만 읽음
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

RESULT_DIR = os.path.join(PROCESSED_DIR, "model_results")

## 2. 결과 파일 읽기

In [ ]:
frames = []

for file_name in sorted(os.listdir(RESULT_DIR)):
    if file_name.endswith("_summary.csv"):
        frames.append(pd.read_csv(os.path.join(RESULT_DIR, file_name), encoding=ENC))

results = pd.concat(frames, ignore_index=True)
print("읽은 결과:", len(frames), "개")

## 3. 구역별 F1 표

In [ ]:
MODEL_ORDER = ["Dummy (전부 0)", "Z-score (지난 값)", "IsolationForest", "RandomForest"]

for zone in results["ZONE"].unique():
    part = results[results["ZONE"] == zone]
    table = part.pivot(index="DATA_FILE", columns="모델", values="F1")
    table = table[MODEL_ORDER]

    print("구역:", zone)
    display(table)

## 4. v5와 v6 비교

v6은 v5에 팀 판단만 더한 데이터임. `v6 최종 입력`은 03의 기본값(공통 움직임 + 10분 이동통계)으로 돌린 결과임. 정답 라벨이 바뀌었으므로 F1 차이는 "모델이 좋아짐"이 아니라 "팀 판단을 넣었을 때 얼마나 맞히나"로 읽음.

In [ ]:
FINAL_NAME = "cal01_base_v6.csv + 공통 움직임 + 10분 통계"  # v6에서 정한 입력

rows = []

for zone in results["ZONE"].unique():
    part = results[results["ZONE"] == zone]
    v5 = part[part["DATA_FILE"] == "cal01_base_v5.csv"]
    v6 = part[part["DATA_FILE"] == "cal01_base_v6.csv"]
    final = part[part["DATA_FILE"] == FINAL_NAME]

    if len(v5) == 0 or len(v6) == 0:
        continue

    for name in MODEL_ORDER:
        f1_v5 = v5.loc[v5["모델"] == name, "F1"].iloc[0]
        f1_v6 = v6.loc[v6["모델"] == name, "F1"].iloc[0]

        f1_final = None
        if len(final) > 0:
            f1_final = round(final.loc[final["모델"] == name, "F1"].iloc[0], 3)

        rows.append({
            "구역": zone,
            "모델": name,
            "v5 F1": round(f1_v5, 3),
            "v6 F1": round(f1_v6, 3),
            "차이": round(f1_v6 - f1_v5, 3),
            "v6 최종 입력 F1": f1_final,
            "v5 LABEL 1": int(v5.loc[v5["모델"] == name, "LABEL 1"].iloc[0]),
            "v6 LABEL 1": int(v6.loc[v6["모델"] == name, "LABEL 1"].iloc[0]),
        })

v5_v6 = pd.DataFrame(rows)
v5_v6

## 5. 전체 결과 (TP, FP, FN 포함)

In [ ]:
results